# Project 3 — LSTM Volatility Forecasting + Binomial Pricing

**Pipeline:**
1. Download Nifty 50 daily data (2010–2024) and engineer features
2. Train a stacked LSTM to forecast 21-day forward vol
3. Price ATM options under historical, LSTM, and implied vol
4. Horse-race: which vol source gives smallest pricing error?
5. Vega sensitivity analysis

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..') / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from data_pipeline import (load_nifty, make_sequences, split_data,
                            normalize_splits, save_processed, load_processed)
from lstm_model import VolLSTM, train, predict, evaluate
from pricing_pipeline import (
    price_option_three_ways,
    vega_sensitivity_analysis,
    backtest_pricing,
    summarise_backtest,
)
from backtest import plot_vol_forecast, plot_vega_surface, plot_pricing_error

print('Torch version:', torch.__version__)

Torch version: 2.11.0


## Phase 1 — Data pipeline

In [2]:
df = load_nifty(start='2010-01-01', end='2024-12-31')
print(df.shape)
df.tail(3)

/Users/adityanagarsekar/Downloads/Courses/StochasticCalculus/Stochastic_Calculus_and_its_Applications_In_Finance_Course_Project/notebooks/../src/data_pipeline.py:41: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(cache_path, index_col=0, parse_dates=True)


TypeError: unsupported operand type(s) for /: 'str' and 'str'

In [ ]:
# Plot realised vol to see vol clusters (sanity check)
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df.index, df['rv21'], label='21d realised vol', lw=1)
ax.plot(df.index, df['target_vol'], label='target (fwd 21d)', lw=1, ls='--', alpha=0.7)
ax.set_title('Nifty 50 — Rolling realised vol')
ax.set_ylabel('Annualised sigma')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
SEQ_LEN = 60
X, y, dates = make_sequences(df, seq_len=SEQ_LEN)
print(f'X: {X.shape}  y: {y.shape}')

splits = split_data(X, y, dates)

# Normalise features — fit on train only, apply to val/test
# This is critical: log_ret (~0.01), ret_sq (~0.0001), rv5/rv21 (~0.15)
# are on wildly different scales and will cause the LSTM to predict a constant
splits, scaler = normalize_splits(splits)

for k, v in splits.items():
    print(f"{k}: {v['X'].shape}  dates {v['dates'].min().date()} – {v['dates'].max().date()}")

# Sanity check: training features should now be ~N(0,1)
flat = splits['train']['X'].reshape(-1, splits['train']['X'].shape[-1])
print(f"\nFeature means (should be ~0): {flat.mean(axis=0).round(3)}")
print(f"Feature stds  (should be ~1): {flat.std(axis=0).round(3)}")

save_processed(splits, scaler)

## Phase 2 — LSTM training

In [ ]:
model, history = train(
    splits['train']['X'], splits['train']['y'],
    splits['val']['X'],   splits['val']['y'],
    epochs=100,
    lr=1e-3,
    batch_size=64,
    patience=15,
    hidden1=64,
    hidden2=32,
    dropout=0.20,
    checkpoint_name='lstm_vol_best.pt',
)

In [ ]:
# Loss curves
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(np.sqrt(history['train_loss']), label='Train RMSE')
ax.plot(np.sqrt(history['val_loss']),   label='Val RMSE')
ax.set_xlabel('Epoch')
ax.set_ylabel('RMSE (annualised sigma)')
ax.set_title('LSTM training curves')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
y_pred_test = predict(model, splits['test']['X'])
y_true_test = splits['test']['y']

metrics = evaluate(y_true_test, y_pred_test)
print('Test RMSE: {:.4f}  MAE: {:.4f}  Dir accuracy: {:.2%}'.format(
    metrics['rmse'], metrics['mae'], metrics['directional_accuracy']))

# Scatter: predicted vs actual
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(y_true_test, y_pred_test, alpha=0.2, s=5)
lims = [min(y_true_test.min(), y_pred_test.min()),
        max(y_true_test.max(), y_pred_test.max())]
ax.plot(lims, lims, 'r--', lw=1)
ax.set_xlabel('Realised vol')
ax.set_ylabel('LSTM forecast')
ax.set_title('Predicted vs Actual vol (test set)')
plt.tight_layout()
plt.savefig('../results/lstm_scatter.png', dpi=150)
plt.show()

In [ ]:
# Time-series forecast plot
plot_vol_forecast(splits['test']['dates'], y_true_test, y_pred_test)

## Phase 3 — Pricing pipeline

In [ ]:
# Vega sensitivity analysis
sigma_range = np.linspace(0.05, 0.60, 100)
S, K, r = 100.0, 100.0, 0.065

df_vega = vega_sensitivity_analysis(S, K, r, sigma_range,
                                     maturities=[7/252, 21/252, 63/252])
plot_vega_surface(df_vega)

# Show price error from 1 ppt vol error at ATM, 1-month
row = df_vega[(df_vega['T_days'] == 21) & (df_vega['sigma'].round(2) == 0.20)].iloc[0]
print(f"ATM 1m: Vega={row['vega']:.3f}, "
      f"1ppt vol error → price error ≈ {row['price_err_1pct']:.4f}")

In [ ]:
# Demonstrate three-way pricing for a sample option
# (in the real backtest below we use the full test set)
sample_idx = 0
test_prices = df.loc[splits['test']['dates'], 'Close'].values
hist_vols   = df.loc[splits['test']['dates'], 'rv21'].values

result = price_option_three_ways(
    S=test_prices[sample_idx],
    K=round(test_prices[sample_idx] / 100) * 100,
    T=21/252,
    r=0.065,
    sigma_hist=hist_vols[sample_idx],
    sigma_lstm=float(y_pred_test[sample_idx]),
)
import json
print(json.dumps(result, indent=2))

# Demonstrate three-way pricing for a sample option
# (in the real backtest below we use the full test set)
sample_idx = 0
test_prices = df.loc[splits['test']['dates'], 'Close'].values
hist_vols   = df.loc[splits['test']['dates'], 'rv21'].values

S_sample = float(test_prices[sample_idx])

result = price_option_three_ways(
    S=S_sample,
    K=round(S_sample / 100) * 100,
    T=21/252,
    r=0.065,
    sigma_hist=float(hist_vols[sample_idx]),
    sigma_lstm=float(y_pred_test[sample_idx]),
)
import json
print(json.dumps(result, indent=2))

In [ ]:
from bsm import bsm_price as _bsm

test_spots = df.loc[splits['test']['dates'], 'Close'].values
test_hist  = df.loc[splits['test']['dates'], 'rv21'].values
test_true  = splits['test']['y']   # forward vol (ground truth)
test_lstm  = y_pred_test

T, r = 21 / 252, 0.065
rng  = np.random.default_rng(42)

# Proxy market prices: true forward vol + 5% noise
proxy_market_prices = np.array([
    _bsm(S, round(S/100)*100, T, r,
         max(test_true[i] * rng.uniform(0.95, 1.05), 0.01))
    for i, S in enumerate(test_spots)
])

df_bt = backtest_pricing(
    prices=test_spots,
    dates=splits['test']['dates'],
    hist_vols=test_hist,
    lstm_vols=test_lstm,
    market_prices=proxy_market_prices,
    T=T,
    r=r,
)

print(summarise_backtest(df_bt))

In [ ]:
plot_pricing_error(df_bt)

# LSTM σ vs implied σ scatter
lstm_rows = df_bt[df_bt['method'] == 'lstm']
impl_rows = df_bt[df_bt['method'] == 'implied']

merged = lstm_rows[['date','sigma']].merge(
    impl_rows[['date','sigma']], on='date', suffixes=('_lstm','_impl')
)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(merged['sigma_impl'], merged['sigma_lstm'], alpha=0.3, s=5)
lims = [0, max(merged['sigma_impl'].max(), merged['sigma_lstm'].max()) * 1.05]
ax.plot(lims, lims, 'r--', lw=1)
ax.set_xlabel('Implied sigma')
ax.set_ylabel('LSTM sigma')
ax.set_title('LSTM forecast vs Implied vol')
plt.tight_layout()
plt.savefig('../results/lstm_vs_implied.png', dpi=150)
plt.show()

In [ ]:
# Save backtest results
df_bt.to_csv('../results/pricing_backtest.csv', index=False)
print('Results saved to results/pricing_backtest.csv')